In [1]:
import os
import pandas as pd
import urllib.request
from pathlib import Path
from tqdm import tqdm
import multiprocessing
from functools import partial

# =============================
# 1. Paths and data loading
# =============================
TRAIN_CSV = "../dataset/train.csv"
TEST_CSV  = "../dataset/test.csv"

IMG_DIR_TRAIN = "../images/train"
IMG_DIR_TEST  = "../images/test"

os.makedirs(IMG_DIR_TRAIN, exist_ok=True)
os.makedirs(IMG_DIR_TEST, exist_ok=True)

train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)

print("Train shape:", train.shape)
print("Test shape:", test.shape)


Train shape: (75000, 4)
Test shape: (75000, 3)


In [4]:
import os
import urllib.request
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

def _save_one(row, savefolder):
    """Download one image: row = (image_link, sample_id)"""
    image_link, sample_id = row
    save_path = os.path.join(savefolder, f"{int(sample_id)}.jpg")

    # Skip if already exists and is non-empty
    if os.path.exists(save_path) and os.path.getsize(save_path) > 0:
        return True  

    try:
        urllib.request.urlretrieve(image_link, save_path)
        if os.path.getsize(save_path) == 0:
            raise RuntimeError("Zero-byte file")
        return True
    except Exception as ex:
        # mark failed with zero-byte file
        try:
            open(save_path, "wb").close()
        except:
            pass
        return False

def download_images_parallel(df, img_dir, max_workers=16):
    pairs = list(zip(df["image_link"].tolist(), df["sample_id"].tolist()))
    ok = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_save_one, row, img_dir): row for row in pairs}
        for future in tqdm(as_completed(futures), total=len(futures)):
            if future.result():
                ok += 1

    total = len(pairs)
    print(f"{img_dir}: downloaded ok={ok}/{total}, failed={total-ok}")


In [5]:
print("Running smoke test (20 images)...")
download_images_parallel(train.sample(20, random_state=42), IMG_DIR_TRAIN, max_workers=16)


Running smoke test (20 images)...


100%|██████████| 20/20 [00:00<00:00, 88862.37it/s]

../images/train: downloaded ok=20/20, failed=0


In [8]:
download_images_parallel(train, IMG_DIR_TRAIN, max_workers=32)


100%|██████████| 75000/75000 [24:04<00:00, 51.94it/s]  

../images/train: downloaded ok=74999/75000, failed=1


In [10]:
download_images_parallel(test, IMG_DIR_TEST, max_workers=16)


100%|██████████| 75000/75000 [00:01<00:00, 67982.99it/s]

../images/test: downloaded ok=74999/75000, failed=1
